# EXP-DENSENET121-TRANSFER-5SEEDS-019 — Final Test + real holdout

**Evaluation only.** Load the five Validation-locked checkpoints from EXP-018.
There is no training call or optimizer update in this notebook. Results are
written to a separate EXP-019 directory. The internal Test has 686 images and
five true classes; the external holdout has 127 images and four true classes.
Both are scored once, with the same five checkpoints and no threshold tuning.

Run all cells in order on Colab. The configuration searches the project
shortcut in the active account's My Drive. If this runtime still has the old
account mounted, set `FORCE_REMOUNT_DRIVE=True`. If the shared project is
read-only, set `DRIVE_RESULTS_ROOT_OVERRIDE` to a writable folder. If EXP-018
is stored elsewhere, set `SOURCE_RESULTS_ROOT_OVERRIDE` to its parent
`current_dataset` folder; the notebook appends the EXP-018 directory name.


In [1]:
# 01. Imports and fixed evaluation protocol
import base64
import csv
import gc
import hashlib
import json
import os
import platform
import random
import shutil
import sys
import time
import zipfile
import zlib
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image, ImageOps
from sklearn.metrics import (accuracy_score, average_precision_score,
                             balanced_accuracy_score, classification_report,
                             confusion_matrix, f1_score, precision_recall_curve,
                             precision_score, recall_score, roc_auc_score, roc_curve)
from tensorflow import keras
from tensorflow.keras import layers

SOURCE_ID = "EXP-DENSENET121-TRANSFER-5SEEDS-018"
EXPERIMENT_ID = "EXP-DENSENET121-TRANSFER-5SEEDS-019"
SEEDS = [42, 123, 2026, 3407, 7777]
CLASSES = ["anthracnose", "healthy", "leaf_miner", "red_rust", "not_cashew_leaf"]
HOLDOUT_CLASSES = ["anthracnose", "healthy", "leaf_miner", "red_rust"]
SPLIT_ZIP = "CashewData_Split (9).zip"
HOLDOUT_ZIP = "RL_Cashew_holdout (2).zip"
SPLIT_SHA256 = "1eb8c46921de19ab53ef8d50e05d8785f94dcb45a5511dcf32e2e7e01516c6f4"
HOLDOUT_SHA256 = "47a5722660e2c8dc7cc04c9ff720e1f889c2481ec99112b41bc0606958a7fdd7"
EXPECTED_TEST_COUNTS = {"anthracnose": 122, "healthy": 118, "leaf_miner": 132,
                        "red_rust": 158, "not_cashew_leaf": 156}
EXPECTED_HOLDOUT_COUNTS = {"anthracnose": 46, "healthy": 20,
                           "leaf_miner": 36, "red_rust": 25}
IMG_SIZE, BATCH_SIZE = 224, 16
NUM_CLASSES = len(CLASSES)
HEAD_DROPOUT = 0.40
WEIGHT_DECAY = 1e-4
FORCE_REMOUNT_DRIVE = False
DRIVE_PROJECT_DIR_OVERRIDE = None
SOURCE_RESULTS_ROOT_OVERRIDE = None
DRIVE_RESULTS_ROOT_OVERRIDE = None
DATASET_ZIP_OVERRIDE = None
HOLDOUT_ZIP_OVERRIDE = None

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 << 20), b""):
            digest.update(block)
    return digest.hexdigest()

def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

print("TensorFlow", tf.__version__, "GPUs", tf.config.list_physical_devices("GPU"))


TensorFlow 2.21.0 GPUs [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
# 02. Mount the current account and locate source/output directories
try:
    from google.colab import drive
except ImportError:
    drive = None
if drive is not None:
    drive.mount("/content/drive", force_remount=FORCE_REMOUNT_DRIVE)
    drive_root = Path("/content/drive/MyDrive")
    if DRIVE_PROJECT_DIR_OVERRIDE is not None:
        project_dir = Path(DRIVE_PROJECT_DIR_OVERRIDE).expanduser()
    else:
        direct = drive_root / "Cashew_Leaf_model_result"
        candidates = [direct] if (direct / "Dataset" / SPLIT_ZIP).is_file() else []
        if not candidates:
            candidates = [p for p in drive_root.iterdir() if p.is_dir() and
                          (p / "Dataset" / SPLIT_ZIP).is_file()]
        if len(candidates) != 1:
            raise FileNotFoundError(
                f"Found {len(candidates)} project shortcuts. Set DRIVE_PROJECT_DIR_OVERRIDE.")
        project_dir = candidates[0]
    default_results = project_dir / "training_results" / "current_dataset"
    source_root = (Path(SOURCE_RESULTS_ROOT_OVERRIDE).expanduser() if
                   SOURCE_RESULTS_ROOT_OVERRIDE else default_results)
    results_root = (Path(DRIVE_RESULTS_ROOT_OVERRIDE).expanduser() if
                    DRIVE_RESULTS_ROOT_OVERRIDE else default_results)
    split_archive = Path(DATASET_ZIP_OVERRIDE) if DATASET_ZIP_OVERRIDE else project_dir / "Dataset" / SPLIT_ZIP
    holdout_archive = Path(HOLDOUT_ZIP_OVERRIDE) if HOLDOUT_ZIP_OVERRIDE else project_dir / "Dataset" / HOLDOUT_ZIP
    cache_root = Path("/content/dataset_cache/exp019")
elif os.name == "nt":
    project_dir = Path(r"D:\Study\KhoaLuanTotNghiep\Resource")
    default_results = project_dir / "training_results" / "current_dataset"
    source_root = Path(SOURCE_RESULTS_ROOT_OVERRIDE) if SOURCE_RESULTS_ROOT_OVERRIDE else default_results
    results_root = Path(DRIVE_RESULTS_ROOT_OVERRIDE) if DRIVE_RESULTS_ROOT_OVERRIDE else default_results
    dataset_dir = Path(r"D:\Study\KhoaLuanTotNghiep\Dataset")
    split_archive = Path(DATASET_ZIP_OVERRIDE) if DATASET_ZIP_OVERRIDE else dataset_dir / SPLIT_ZIP
    holdout_archive = Path(HOLDOUT_ZIP_OVERRIDE) if HOLDOUT_ZIP_OVERRIDE else dataset_dir / HOLDOUT_ZIP
    cache_root = project_dir / "dataset_cache" / "exp019"
else:
    raise RuntimeError("EXP-019 supports Colab or Windows; set explicit paths for another runtime.")

source_dir = source_root / SOURCE_ID
out_dir = results_root / EXPERIMENT_ID
aggregate_dir = out_dir / "aggregate"
figures_dir = out_dir / "figures"
for directory in (out_dir, aggregate_dir, figures_dir):
    directory.mkdir(parents=True, exist_ok=True)
probe = out_dir / ".write_probe"
try:
    probe.write_text("ok", encoding="utf-8")
finally:
    probe.unlink(missing_ok=True)
if not source_dir.is_dir():
    raise FileNotFoundError(f"EXP-018 result directory not found: {source_dir}")
for path in (split_archive, holdout_archive):
    if not path.is_file():
        raise FileNotFoundError(path)
print("EXP-018:", source_dir)
print("EXP-019:", out_dir)
print("Split ZIP:", split_archive)
print("Holdout ZIP:", holdout_archive)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
EXP-018: /content/drive/MyDrive/Cashew_Leaf_model_result/training_results/current_dataset/EXP-DENSENET121-TRANSFER-5SEEDS-018
EXP-019: /content/drive/MyDrive/Cashew_Leaf_model_result/training_results/current_dataset/EXP-DENSENET121-TRANSFER-5SEEDS-019
Split ZIP: /content/drive/MyDrive/Cashew_Leaf_model_result/Dataset/CashewData_Split (9).zip
Holdout ZIP: /content/drive/MyDrive/Cashew_Leaf_model_result/Dataset/RL_Cashew_holdout (2).zip


## Verify the EXP-018 lock and the two revised archives

Any SHA-256 or checkpoint mismatch stops the evaluation before inference.


In [3]:
# 03. Verify recipe, selection lock, five checkpoints and archive identities
config_path = source_dir / "experiment_config.json"
lock_path = source_dir / "validation_selection_lock.json"
config = json.loads(config_path.read_text(encoding="utf-8"))
lock = json.loads(lock_path.read_text(encoding="utf-8"))
if config.get("experiment_id") != SOURCE_ID or lock.get("experiment_id") != SOURCE_ID:
    raise ValueError("Source experiment identity mismatch")
if config.get("dataset", {}).get("archive_sha256", "").lower() != SPLIT_SHA256:
    raise ValueError("Source recipe uses a different dataset ZIP")
if lock.get("dataset_sha256", "").lower() != SPLIT_SHA256:
    raise ValueError("Validation lock uses a different dataset ZIP")
if lock.get("config_sha256", "").lower() != sha256_file(config_path):
    raise ValueError("Source config changed after Validation lock")
if config.get("dataset", {}).get("class_names") != CLASSES or lock.get("class_names") != CLASSES:
    raise ValueError("Class order changed")
if config.get("seeds") != SEEDS or lock.get("seeds") != SEEDS:
    raise ValueError("Five-seed protocol changed")
if lock.get("dataset_manifest_sha256") != config["dataset"]["manifest_sha256"]:
    raise ValueError("Source manifest fingerprint mismatch")
if sha256_file(split_archive) != SPLIT_SHA256 or sha256_file(holdout_archive) != HOLDOUT_SHA256:
    raise ValueError("Split 9 or Holdout 2 archive SHA-256 mismatch")
checkpoint_paths = {seed: source_dir / f"seed_{seed}" / "best.weights.h5" for seed in SEEDS}
checkpoint_hashes = {}
for seed, path in checkpoint_paths.items():
    if not path.is_file():
        raise FileNotFoundError(path)
    actual = sha256_file(path)
    if actual.lower() != lock["selected_checkpoint_sha256"][str(seed)].lower():
        raise ValueError(f"Checkpoint changed: seed {seed}")
    checkpoint_hashes[str(seed)] = actual
deployment_seed = int(lock["deployment_seed"])
if deployment_seed not in SEEDS:
    raise ValueError("Invalid deployment seed in source lock")
validation_csv = source_dir / "aggregate" / "validation_results_5seeds.csv"
if sha256_file(validation_csv) != lock["validation_sha256"]:
    raise ValueError("Validation results changed after lock")
validation_rows = pd.read_csv(validation_csv)
if validation_rows["seed"].tolist() != SEEDS:
    raise ValueError("Source Validation results are incomplete")
print("Source lock PASS; deployment seed:", deployment_seed)


Source lock PASS; deployment seed: 123


In [4]:
# 04. Safe extraction and exact manifest verification
def extract_frozen_zip(archive_path, target_parent, root_name, archive_sha):
    target = target_parent / root_name
    marker = target_parent / ".archive_sha256"
    if target_parent.exists():
        if not marker.is_file() or marker.read_text(encoding="utf-8").strip() != archive_sha:
            raise RuntimeError(f"Unverified extraction cache: {target_parent}")
    else:
        staging = target_parent.with_name(target_parent.name + "_staging")
        if staging.exists():
            raise RuntimeError(f"Partial extraction requires review: {staging}")
        staging.mkdir(parents=True)
        with zipfile.ZipFile(archive_path) as archive:
            for member in archive.infolist():
                name = member.filename.replace("\\", "/")
                parts = Path(name).parts
                resolved = (staging / name).resolve()
                if not parts or parts[0] != root_name or staging.resolve() not in resolved.parents:
                    raise ValueError(f"Unsafe or unexpected ZIP member: {member.filename}")
            archive.extractall(staging)
        (staging / ".archive_sha256").write_text(archive_sha, encoding="utf-8")
        staging.rename(target_parent)
    if not target.is_dir():
        raise FileNotFoundError(target)
    return target

split_root = extract_frozen_zip(split_archive, cache_root / "split09", "CashewData_Split", SPLIT_SHA256)
holdout_root = extract_frozen_zip(holdout_archive, cache_root / "holdout02", "RL_Cashew_holdout", HOLDOUT_SHA256)
source_manifest_path = source_dir / "dataset_manifest.csv"
source_manifest = pd.read_csv(source_manifest_path, dtype={"sha256": str, "relative_path": str})
needed_columns = {"relative_path", "label", "sha256", "split", "class"}
if not needed_columns.issubset(source_manifest.columns) or len(source_manifest) != 7045:
    raise ValueError("EXP-018 source manifest is missing or incomplete")
manifest_records = [{"relative_path": str(row.relative_path), "label": int(row.label),
                     "sha256": str(row.sha256)}
                    for row in source_manifest.itertuples(index=False)]
manifest_digest = hashlib.sha256(json.dumps(
    manifest_records, sort_keys=True, separators=(",", ":")).encode()).hexdigest()
if manifest_digest != lock["dataset_manifest_sha256"]:
    raise ValueError("EXP-018 source manifest content changed")
reference_hashes = set()
for row in source_manifest.itertuples(index=False):
    relative = Path(row.relative_path)
    if relative.is_absolute() or ".." in relative.parts:
        raise ValueError(f"Unsafe source manifest path: {row.relative_path}")
    image = split_root / relative
    if not image.is_file() or sha256_file(image) != row.sha256:
        raise ValueError(f"Split 9 extracted image changed: {row.relative_path}")
    reference_hashes.add(row.sha256)
test_manifest = source_manifest[source_manifest["split"] == "test"].copy().reset_index(drop=True)
if test_manifest["class"].value_counts().to_dict() != EXPECTED_TEST_COUNTS:
    raise ValueError("Split 9 Test class counts changed")
test_manifest["image_path"] = test_manifest["relative_path"].map(lambda x: str(split_root / x))
val_manifest = source_manifest[source_manifest["split"] == "val"].head(2).copy()
val_manifest["image_path"] = val_manifest["relative_path"].map(lambda x: str(split_root / x))
if not test_manifest["image_path"].map(lambda x: Path(x).is_file()).all():
    raise FileNotFoundError("Some Test images are missing")

holdout_rows = []
for name in HOLDOUT_CLASSES:
    folder = holdout_root / name
    files = sorted(p for p in folder.iterdir() if p.is_file())
    if len(files) != EXPECTED_HOLDOUT_COUNTS[name]:
        raise ValueError(f"Holdout class count changed: {name}")
    for image in files:
        with Image.open(image) as opened:
            opened.verify()
        digest = sha256_file(image)
        if digest in reference_hashes:
            raise ValueError(f"Holdout image overlaps Split 9: {image}")
        holdout_rows.append({"relative_path": image.relative_to(holdout_root).as_posix(),
                             "image_path": str(image), "class": name,
                             "label": CLASSES.index(name), "sha256": digest})
holdout_manifest = pd.DataFrame(holdout_rows)
if len(holdout_manifest) != 127 or holdout_manifest["sha256"].duplicated().any():
    raise ValueError("Holdout 2 contains missing or duplicate images")
test_manifest.to_csv(out_dir / "test_dataset_manifest.csv", index=False)
holdout_manifest.to_csv(out_dir / "holdout_dataset_manifest.csv", index=False)
save_json(out_dir / "dataset_audit.json", {
    "split_zip_sha256": SPLIT_SHA256, "source_manifest_sha256": manifest_digest,
    "test_counts": EXPECTED_TEST_COUNTS, "holdout_zip_sha256": HOLDOUT_SHA256,
    "holdout_counts": EXPECTED_HOLDOUT_COUNTS, "holdout_exact_overlap": 0,
    "limitation": "Near duplicates and shared leaf/tree/session IDs need separate review.",
})
print("Dataset audit PASS: Test", len(test_manifest), "Holdout", len(holdout_manifest))


Dataset audit PASS: Test 686 Holdout 127


## Rebuild the EXP-018 model graph and load frozen weights

The model definition is copied from EXP-018. `weights=None` only avoids
redownloading ImageNet weights; the trained checkpoint is loaded immediately.


In [5]:

# ============================================================
# 04. REPRODUCIBILITY
# ============================================================

def set_global_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

    # Đồng bộ Python / NumPy / TensorFlow theo Keras.
    try:
        keras.utils.set_random_seed(seed)
    except Exception:
        pass

    print(f"Global seed set to: {seed}")


try:
    tf.config.experimental.enable_op_determinism()
    print("Deterministic operations: ENABLED")
except Exception as e:
    print("Deterministic operations: unavailable")
    print("Reason:", e)

# 08. ImageNet backbone, frozen BN, and smoke test
import tempfile
from PIL import ImageOps

AUGMENTATION_CONFIG = {"horizontal_flip": True, "rotation_factor": 0.08,
                       "zoom_factor": 0.10, "contrast_factor": 0.10,
                       "brightness": False}

def densenet_preprocessing(inputs):
    x = layers.Rescaling(1.0 / 255.0, name="densenet_rescale")(inputs)
    return layers.Normalization(mean=[0.485, 0.456, 0.406],
                                variance=[0.229**2, 0.224**2, 0.225**2],
                                axis=-1, name="densenet_preprocess")(x)

def build_transfer_model(seed, weights="imagenet"):
    set_global_seed(seed)
    inputs = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="raw_rgb")
    augmented = keras.Sequential([
        layers.RandomFlip("horizontal", seed=seed),
        layers.RandomRotation(0.08, seed=seed),
        layers.RandomZoom(0.10, seed=seed),
        layers.RandomContrast(0.10, seed=seed),
    ], name=f"train_only_augmentation_seed_{seed}")(inputs)
    x = densenet_preprocessing(augmented)
    backbone = keras.applications.DenseNet121(include_top=False, weights=weights,
                                               input_shape=(IMG_SIZE, IMG_SIZE, 3), pooling="avg")
    backbone.trainable = False
    features = backbone(x, training=False)  # Keep BN moving statistics fixed in both stages.
    x = layers.Dropout(HEAD_DROPOUT, seed=seed)(features)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32",
                           name="disease_output")(x)
    return keras.Model(inputs, outputs, name=f"DenseNet121_Transfer_seed_{seed}"), backbone

def set_backbone_stage(backbone, stage):
    if stage == "stage1":
        backbone.trainable = False
    elif stage == "stage2":
        backbone.trainable = True
        for layer in backbone.layers:
            layer.trainable = (layer.name.startswith("conv5_") and
                               not isinstance(layer, layers.BatchNormalization))
        if not any(layer.trainable and layer.weights for layer in backbone.layers):
            raise RuntimeError("No weighted conv5_* layer was unfrozen")
        if any(layer.trainable for layer in backbone.layers
               if isinstance(layer, layers.BatchNormalization)):
            raise RuntimeError("A backbone BatchNorm layer is trainable")
    else:
        raise ValueError(stage)
    return int(sum(np.prod(v.shape) for v in backbone.trainable_weights))

def compile_stage(model, lr):
    model.compile(optimizer=keras.optimizers.AdamW(learning_rate=lr,
                                                  weight_decay=WEIGHT_DECAY),
                  loss=keras.losses.SparseCategoricalCrossentropy(),
                  metrics=["accuracy"], steps_per_execution=1, jit_compile=False)

def save_json(path, value):
    path.write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding="utf-8")

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def load_locked_model(seed):
    keras.backend.clear_session()
    gc.collect()
    model, backbone = build_transfer_model(seed, weights=None)
    selection = json.loads((source_dir / f"seed_{seed}" / "stage_selection.json").read_text(encoding="utf-8"))
    stage = selection["selected_stage"]
    if stage not in {"stage1", "stage2"}:
        raise ValueError(f"Unexpected selected stage for seed {seed}: {stage}")
    if selection.get("checkpoint_sha256", "").lower() != checkpoint_hashes[str(seed)]:
        raise ValueError(f"Stage selection/checkpoint fingerprint mismatch: seed {seed}")
    set_backbone_stage(backbone, stage)
    model.load_weights(checkpoint_paths[seed])
    return model

def make_eval_dataset(frame):
    paths = frame["image_path"].astype(str).to_numpy()
    labels = frame["label"].to_numpy(dtype=np.int32)
    def decode(path, label):
        raw = tf.io.read_file(path)
        image = tf.io.decode_image(raw, channels=3, expand_animations=False)
        image.set_shape([None, None, 3])
        image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE], method="bilinear")
        return tf.cast(image, tf.float32), label
    return tf.data.Dataset.from_tensor_slices((paths, labels)).map(
        decode, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# Verify the selected seed model against the source .keras deployment package.
source_deployment = source_dir / "model_package" / "deployment_model.keras"
if not source_deployment.is_file():
    raise FileNotFoundError(source_deployment)
selected_model = load_locked_model(deployment_seed)
reference_model = keras.models.load_model(source_deployment, compile=False)
sample = next(iter(make_eval_dataset(val_manifest)))[0]
if not np.allclose(selected_model(sample, training=False).numpy(),
                   reference_model(sample, training=False).numpy(), atol=1e-5):
    raise RuntimeError("Rebuilt checkpoint disagrees with EXP-018 deployment model")
del selected_model, reference_model
keras.backend.clear_session(); gc.collect()
print("Checkpoint/model parity PASS")


Deterministic operations: ENABLED
Global seed set to: 123
Checkpoint/model parity PASS


In [6]:
# 06. Freeze evaluation inputs before any Test or holdout inference
evaluation_lock = {
    "experiment_id": EXPERIMENT_ID, "source_experiment_id": SOURCE_ID,
    "source_config_sha256": sha256_file(config_path),
    "source_validation_lock_sha256": sha256_file(lock_path),
    "source_validation_results_sha256": sha256_file(validation_csv),
    "source_manifest_sha256": manifest_digest,
    "split_zip_sha256": SPLIT_SHA256, "holdout_zip_sha256": HOLDOUT_SHA256,
    "selected_checkpoint_sha256": checkpoint_hashes,
    "seeds": SEEDS, "deployment_seed": deployment_seed,
    "class_order": CLASSES, "selection_source": "EXP-018 Validation only",
    "test_images": 686, "holdout_images": 127,
}
evaluation_lock_path = out_dir / "evaluation_lock.json"
if evaluation_lock_path.exists():
    if json.loads(evaluation_lock_path.read_text(encoding="utf-8")) != evaluation_lock:
        raise RuntimeError("EXP-019 evaluation lock changed; do not mix datasets or checkpoints")
else:
    save_json(evaluation_lock_path, evaluation_lock)
print("Evaluation lock PASS")


Evaluation lock PASS


## Final Test and external holdout

The five checkpoints are fixed. Holdout macro metrics use only its four
supported true classes. Every prediction retains all five class probabilities.


In [7]:
# 07. Evaluate every frozen checkpoint once and export standard atomic outputs
PROB_COLUMNS = [f"prob_{name}" for name in CLASSES]

def write_matrix(matrix, output, normalized):
    suffix = "normalized" if normalized else "raw"
    pd.DataFrame(matrix, index=CLASSES, columns=CLASSES).to_csv(
        output / f"confusion_matrix_{suffix}.csv")
    fig, ax = plt.subplots(figsize=(8, 7), facecolor="white")
    ax.imshow(matrix, cmap="Blues", vmin=0, vmax=1 if normalized else max(1, matrix.max()))
    ax.set_xticks(range(NUM_CLASSES), CLASSES, rotation=40, ha="right")
    ax.set_yticks(range(NUM_CLASSES), CLASSES)
    for r in range(NUM_CLASSES):
        for c in range(NUM_CLASSES):
            ax.text(c, r, f"{matrix[r,c]:.2f}" if normalized else str(int(matrix[r,c])),
                    ha="center", va="center", fontsize=9)
    ax.set(xlabel="Predicted", ylabel="True", title=f"{output.parent.name} / {output.name}")
    fig.tight_layout(); fig.savefig(output / f"confusion_matrix_{suffix}.png", dpi=180); plt.close(fig)

def write_calibration(predictions, output):
    confidence = predictions["confidence"].to_numpy(dtype=float)
    correct = predictions["correct"].to_numpy(dtype=bool)
    probs = predictions[PROB_COLUMNS].to_numpy(dtype=float)
    truth = predictions["true_class_idx"].to_numpy(dtype=int)
    rows = []
    edges = np.linspace(0, 1, 11)
    for i in range(10):
        mask = (confidence >= edges[i]) & (confidence < edges[i+1] if i < 9 else confidence <= 1)
        n = int(mask.sum())
        rows.append({"bin": i, "n": n, "lower": edges[i], "upper": edges[i+1],
                     "accuracy": float(correct[mask].mean()) if n else np.nan,
                     "mean_confidence": float(confidence[mask].mean()) if n else np.nan})
    bins = pd.DataFrame(rows)
    bins.to_csv(output / "calibration_bins.csv", index=False)
    ece = float(sum(r["n"] / len(predictions) * abs(r["accuracy"] - r["mean_confidence"])
                    for r in rows if r["n"]))
    brier = float(np.mean(np.sum((probs - np.eye(NUM_CLASSES)[truth]) ** 2, axis=1)))
    save_json(output / "calibration_metrics.json", {"ece_10_bins": ece,
              "multiclass_brier": brier, "n": len(predictions),
              "note": "Diagnostic only; no threshold or calibration fit."})
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    used = bins[bins["n"] > 0]
    axes[0].plot([0, 1], [0, 1], "--", color="gray")
    axes[0].plot(used["mean_confidence"], used["accuracy"], "o-")
    axes[0].set(xlim=(0, 1), ylim=(0, 1), xlabel="Confidence", ylabel="Accuracy",
                title=f"Reliability; ECE={ece:.3f}")
    axes[1].hist(confidence[correct], bins=edges, alpha=.7, label="Correct")
    axes[1].hist(confidence[~correct], bins=edges, alpha=.7, label="Incorrect")
    axes[1].set(xlabel="Confidence", ylabel="Images", title="Confidence distribution")
    axes[1].legend(); fig.tight_layout()
    fig.savefig(output / "reliability_confidence.png", dpi=180); plt.close(fig)
    return ece, brier

def write_roc_pr(predictions, output):
    truth = predictions["true_class_idx"].to_numpy(dtype=int)
    probs = predictions[PROB_COLUMNS].to_numpy(dtype=float)
    rows = []
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for index, name in enumerate(CLASSES):
        binary = (truth == index).astype(int)
        if binary.sum() in (0, len(binary)):
            rows.append({"class": name, "support": int(binary.sum()), "roc_auc": np.nan,
                         "average_precision": np.nan, "status": "UNDEFINED"})
            continue
        fpr, tpr, _ = roc_curve(binary, probs[:, index])
        precision, recall, _ = precision_recall_curve(binary, probs[:, index])
        auc_score = float(roc_auc_score(binary, probs[:, index]))
        ap = float(average_precision_score(binary, probs[:, index]))
        rows.append({"class": name, "support": int(binary.sum()),
                     "roc_auc": auc_score, "average_precision": ap, "status": "OK"})
        axes[0].plot(fpr, tpr, label=f"{name} {auc_score:.3f}")
        axes[1].plot(recall, precision, label=f"{name} {ap:.3f}")
    axes[0].plot([0, 1], [0, 1], "--", color="gray")
    axes[0].set(xlabel="False positive rate", ylabel="True positive rate", title="ROC OVR")
    axes[1].set(xlabel="Recall", ylabel="Precision", title="PR OVR")
    for ax in axes:
        ax.legend(fontsize=8); ax.grid(alpha=.2)
    fig.tight_layout(); fig.savefig(output / "roc_pr_ovr.png", dpi=180); plt.close(fig)
    pd.DataFrame(rows).to_csv(output / "roc_pr_ovr_metrics.csv", index=False)

def score_split(split_name, frame, seed, model, output):
    dataset = make_eval_dataset(frame)
    _ = model.predict(dataset.take(1), verbose=0)
    started = time.perf_counter()
    probabilities = np.asarray(model.predict(dataset, verbose=0))
    elapsed = time.perf_counter() - started
    y_true = frame["label"].to_numpy(dtype=int)
    if probabilities.shape != (len(frame), NUM_CLASSES) or not np.isfinite(probabilities).all():
        raise RuntimeError("Prediction matrix has wrong shape or non-finite values")
    if not np.allclose(probabilities.sum(axis=1), 1, atol=1e-4):
        raise RuntimeError("Prediction probabilities do not sum to one")
    y_pred = probabilities.argmax(axis=1)
    supported = [CLASSES.index(x) for x in (CLASSES if split_name == "test" else HOLDOUT_CLASSES)]
    report = classification_report(y_true, y_pred, labels=np.arange(NUM_CLASSES),
                                   target_names=CLASSES, output_dict=True, zero_division=0)
    report_frame = pd.DataFrame(report).T
    if split_name == "real_holdout":
        report_frame.loc["not_cashew_leaf", ["precision", "recall", "f1-score"]] = np.nan
    report_frame.to_csv(output / "classification_report.csv")
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(NUM_CLASSES))
    cm_norm = confusion_matrix(y_true, y_pred, labels=np.arange(NUM_CLASSES), normalize="true")
    write_matrix(cm, output, False)
    write_matrix(cm_norm, output, True)
    relative = frame["relative_path"].astype(str).to_numpy()
    predictions = pd.DataFrame({
        "experiment_id": EXPERIMENT_ID, "source_experiment_id": SOURCE_ID,
        "architecture": "DenseNet121", "training_mode": "ImageNet Transfer, two-stage",
        "dataset_version": ("Cashew_dataV09_more_real" if split_name == "test" else
                            "RL_Cashew_holdout_02"),
        "dataset_sha256": SPLIT_SHA256 if split_name == "test" else HOLDOUT_SHA256,
        "split": split_name, "seed": seed, "image_id": relative, "relative_path": relative,
        "true_label": [CLASSES[i] for i in y_true],
        "predicted_label": [CLASSES[i] for i in y_pred],
        "true_class_idx": y_true, "predicted_class_idx": y_pred,
        "correct": y_true == y_pred, "confidence": probabilities.max(axis=1),
    })
    for index, name in enumerate(CLASSES):
        predictions[f"prob_{name}"] = probabilities[:, index]
    predictions.to_csv(output / "predictions.csv", index=False)
    predictions.loc[~predictions["correct"]].to_csv(output / "error_cases.csv", index=False)
    ece, brier = write_calibration(predictions, output)
    write_roc_pr(predictions, output)
    disease = np.isin(y_true, [CLASSES.index(x) for x in ("anthracnose", "leaf_miner", "red_rust")])
    false_healthy = int(np.sum(disease & (y_pred == CLASSES.index("healthy"))))
    metrics = {
        "experiment_id": EXPERIMENT_ID, "source_experiment_id": SOURCE_ID,
        "split": split_name, "seed": seed, "n": len(frame),
        "macro_class_count": len(supported), "supported_true_classes": [CLASSES[i] for i in supported],
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_precision": float(precision_score(y_true, y_pred, labels=supported,
                                                    average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_true, y_pred, labels=supported,
                                              average="macro", zero_division=0)),
        "macro_f1": float(f1_score(y_true, y_pred, labels=supported,
                                      average="macro", zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "loss": float(-np.log(np.clip(probabilities[np.arange(len(y_true)), y_true], 1e-7, 1)).mean()),
        "false_healthy_count": false_healthy, "disease_images": int(disease.sum()),
        "false_healthy_rate": float(false_healthy / disease.sum()),
        "ece_10_bins": ece, "multiclass_brier": brier,
        "prediction_seconds_including_input_pipeline": elapsed,
        "inference_ms_per_image_batch16": 1000 * elapsed / len(frame),
        "checkpoint_sha256": checkpoint_hashes[str(seed)],
    }
    save_json(output / "metrics.json", metrics)
    return metrics

all_metrics = {}
for split_name, frame in (("test", test_manifest), ("real_holdout", holdout_manifest)):
    rows = []
    for seed in SEEDS:
        output = out_dir / f"seed_{seed}" / split_name
        marker = output / "evaluation_completed.json"
        if marker.is_file():
            saved = json.loads(marker.read_text(encoding="utf-8"))
            if saved.get("evaluation_lock_sha256") != sha256_file(evaluation_lock_path):
                raise RuntimeError(f"Completed {split_name} seed {seed} has a different lock")
            metrics = json.loads((output / "metrics.json").read_text(encoding="utf-8"))
            if not (output / "predictions.csv").is_file():
                raise RuntimeError(f"Completed {split_name} seed {seed} is missing predictions")
            print("Reusing completed", split_name, seed)
        else:
            if output.exists() and any(output.iterdir()):
                raise RuntimeError(f"Partial evaluation requires inspection: {output}")
            output.mkdir(parents=True, exist_ok=True)
            model = load_locked_model(seed)
            metrics = score_split(split_name, frame, seed, model, output)
            del model
            keras.backend.clear_session(); gc.collect()
            save_json(marker, {"evaluation_lock_sha256": sha256_file(evaluation_lock_path),
                               "checkpoint_sha256": checkpoint_hashes[str(seed)],
                               "split": split_name, "seed": seed,
                               "completed_at_utc": datetime.now(timezone.utc).isoformat()})
        rows.append(metrics)
    if [row["seed"] for row in rows] != SEEDS:
        raise RuntimeError("Five-seed evaluation incomplete")
    all_metrics[split_name] = pd.DataFrame(rows)
    all_metrics[split_name].to_csv(aggregate_dir / f"{split_name}_results_5seeds.csv", index=False)
print("Final Test and holdout inference complete")


Global seed set to: 42
Global seed set to: 123
Global seed set to: 2026
Global seed set to: 3407
Global seed set to: 7777
Global seed set to: 42


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


Global seed set to: 123
Global seed set to: 2026


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


Global seed set to: 3407


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


Global seed set to: 7777


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


Final Test and holdout inference complete


In [8]:
# 08. Aggregate summaries and visual panels
for split_name in ("test", "real_holdout"):
    results = all_metrics[split_name]
    metric_names = ["accuracy", "macro_precision", "macro_recall", "macro_f1",
                    "balanced_accuracy", "loss", "false_healthy_rate",
                    "ece_10_bins", "multiclass_brier", "inference_ms_per_image_batch16"]
    summary = pd.DataFrame([{"metric": name, "mean": float(results[name].mean()),
                             "sample_sd": float(results[name].std(ddof=1)),
                             "n_seeds": len(SEEDS), "images_per_seed": int(results.iloc[0]["n"])}
                            for name in metric_names])
    summary.to_csv(aggregate_dir / f"{split_name}_mean_std.csv", index=False)
    per_class = []
    for seed in SEEDS:
        report = pd.read_csv(out_dir / f"seed_{seed}" / split_name /
                             "classification_report.csv", index_col=0)
        for name in CLASSES:
            per_class.append({"split": split_name, "seed": seed, "class": name,
                              "support": int(report.loc[name, "support"]),
                              "precision": report.loc[name, "precision"],
                              "recall": report.loc[name, "recall"],
                              "f1": report.loc[name, "f1-score"]})
    per_class = pd.DataFrame(per_class)
    per_class.to_csv(aggregate_dir / f"per_class_{split_name}_by_seed.csv", index=False)
    per_class.groupby("class")[["precision", "recall", "f1"]].agg(["mean", "std"]).to_csv(
        aggregate_dir / f"per_class_{split_name}_mean_std.csv")
    fig, ax = plt.subplots(figsize=(9, 4.5))
    primary = summary[summary["metric"].isin(["accuracy", "macro_recall", "macro_f1", "balanced_accuracy"])]
    ax.bar(primary["metric"], primary["mean"], yerr=primary["sample_sd"], capsize=5)
    ax.set(ylim=(0, 1.05), ylabel="Score", title=f"{split_name}: mean ± sample SD across five seeds")
    fig.tight_layout(); fig.savefig(figures_dir / f"{split_name}_metrics_mean_seed_sd.png", dpi=180); plt.close(fig)
    heat = per_class.pivot(index="class", columns="seed", values="f1").reindex(CLASSES)
    fig, ax = plt.subplots(figsize=(9, 4.5))
    image = ax.imshow(heat, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(SEEDS)), SEEDS)
    ax.set_yticks(range(NUM_CLASSES), CLASSES)
    ax.set(xlabel="Seed", title=f"{split_name}: per-class F1")
    for r in range(NUM_CLASSES):
        for c in range(len(SEEDS)):
            value = heat.iloc[r, c]
            ax.text(c, r, f"{value:.2f}" if np.isfinite(value) else "N/A",
                    ha="center", va="center")
    fig.colorbar(image, ax=ax, label="F1")
    fig.tight_layout(); fig.savefig(figures_dir / f"{split_name}_per_class_f1_heatmap.png", dpi=180); plt.close(fig)
    for normalized in (False, True):
        suffix = "normalized" if normalized else "raw"
        fig, axes = plt.subplots(1, len(SEEDS), figsize=(27, 5.5))
        for ax, seed in zip(axes, SEEDS):
            matrix = pd.read_csv(out_dir / f"seed_{seed}" / split_name /
                                 f"confusion_matrix_{suffix}.csv", index_col=0).to_numpy()
            ax.imshow(matrix, cmap="Blues", vmin=0, vmax=1 if normalized else max(1, matrix.max()))
            ax.set_xticks(range(NUM_CLASSES), CLASSES, rotation=50, ha="right")
            ax.set_yticks(range(NUM_CLASSES), CLASSES)
            ax.set_title(f"seed {seed}")
            for r in range(NUM_CLASSES):
                for c in range(NUM_CLASSES):
                    ax.text(c, r, f"{matrix[r,c]:.2f}" if normalized else str(int(matrix[r,c])),
                            ha="center", va="center", fontsize=8)
        fig.tight_layout()
        fig.savefig(figures_dir / f"{split_name}_confusion_{suffix}_5seeds.png", dpi=180)
        plt.close(fig)

gap = pd.DataFrame({"seed": SEEDS,
                    "test_5class_macro_f1": all_metrics["test"]["macro_f1"].to_numpy(),
                    "holdout_4class_macro_f1": all_metrics["real_holdout"]["macro_f1"].to_numpy()})
gap.to_csv(aggregate_dir / "domain_gap_descriptive.csv", index=False)
ax = gap.set_index("seed").plot.bar(figsize=(9, 4.5), ylim=(0, 1), rot=0,
    title="Internal Test vs external holdout (different class support)")
ax.set_ylabel("Macro F1"); ax.figure.tight_layout()
ax.figure.savefig(figures_dir / "domain_gap_descriptive.png", dpi=180); plt.close(ax.figure)
print("Aggregate tables and figures complete")


Aggregate tables and figures complete


In [9]:
# 09. Report, manifest and archives; source checkpoints included in FULL ZIP
import base64
import zlib

readme = f"""# {EXPERIMENT_ID}

Evaluation-only continuation of {SOURCE_ID}. No training or selection on Test/holdout.
Source Validation lock: `{sha256_file(lock_path)}`. Deployment seed {deployment_seed}.
Split 9 SHA-256 `{SPLIT_SHA256}`; holdout 2 SHA-256 `{HOLDOUT_SHA256}`.
Test: 686 images, five true classes. Holdout: 127 images, four true classes.
The two macro metrics use different class support. `not_cashew_leaf` holdout
F1 is undefined because it has zero true support.

Per-seed outputs: `seed_*/test|real_holdout/` contains metrics, standard
predictions with five probabilities, errors, reports, confusion CSV/PNG,
calibration ECE/Brier and ROC/PR one-vs-rest. Aggregate summaries and panels
are in `aggregate/` and `figures/`. SD is sample SD across five seeds, not an
image-sampling confidence interval. Prediction timing is batch {BATCH_SIZE}
including data loading after one-batch warm-up, and should not be compared
across different devices as a deployment latency benchmark.

`FULL.zip` adds the five EXP-018 source checkpoints and deployment model;
`ANALYSIS.zip` contains reports, notebook and figures without model binaries.
The source EXP-018 result directory remains unmodified.
"""
(out_dir / "README.md").write_text(readme, encoding="utf-8")
save_json(out_dir / "execution_status.json", {
    "experiment_id": EXPERIMENT_ID, "source_experiment_id": SOURCE_ID,
    "training": "REUSED_EXP018", "final_test": "COMPLETE", "real_holdout": "COMPLETE",
    "source_validation_lock_sha256": sha256_file(lock_path),
    "evaluation_lock_sha256": sha256_file(evaluation_lock_path),
})
required = ["README.md", "execution_status.json", "evaluation_lock.json", "dataset_audit.json",
            "test_dataset_manifest.csv", "holdout_dataset_manifest.csv",
            "aggregate/test_results_5seeds.csv", "aggregate/test_mean_std.csv",
            "aggregate/real_holdout_results_5seeds.csv", "aggregate/real_holdout_mean_std.csv"]
for seed in SEEDS:
    for split_name in ("test", "real_holdout"):
        required += [f"seed_{seed}/{split_name}/{name}" for name in
                     ("metrics.json", "predictions.csv", "classification_report.csv",
                      "confusion_matrix_raw.csv", "confusion_matrix_raw.png",
                      "confusion_matrix_normalized.csv", "confusion_matrix_normalized.png",
                      "calibration_metrics.json", "roc_pr_ovr_metrics.csv",
                      "evaluation_completed.json")]
missing = [name for name in required if not (out_dir / name).is_file() or
           (out_dir / name).stat().st_size == 0]
save_json(out_dir / "audit_required_artifacts.json", {
    "experiment_id": EXPERIMENT_ID, "required_count": len(required),
    "missing": missing, "status": "PASS" if not missing else "FAIL"})
if missing:
    raise RuntimeError(f"Required evaluation artifacts missing: {missing}")

NOTEBOOK_SOURCE_ZLIB_B64 = "eNrNfe1y20iS4KvgNBE7oIeCRFqyZfZyI2SJbuvalhSS3HPTEgMGQVDCGCS4ACiJ1mnjHuEi7v7cY9zv/XdxL7L3JJcfVagPABTlj43t3ekWgUJWVlZWVmZWZtbDRhglSb7Ru3ygv/xiOY82ehvTIPsyTu9mG+2NaVQE46AINnoPj+2NPF1kIbS43PiTM/gvp5uHg+PzwfHgotPtbF6c7R+fvxucbe6eDwaH55vbnTfO//tv/9N5F8+CxLmI8sL5i5NF8PdNmozTRXGF8OlfL14MboNkERRxOnPSWbL0XrxwPqTB2CluImcS30bO70ESj6nBZpKGX6KxE95E4Zd5Gs+K3Jlk6ZQQ2u7seQTy4ibKIifOnVnqFFkQz+LZtRMGSeKkmZPOi3gaf40yZzEHoNBuBj1R4yIapekXzzmL8kUCkIMsInh3WVwUETRLncDJo3mQ4Xfc5RtnHGdRWKTZ0nOgYwBXRFk56psgd17tvXLiaXAdAcTZmCDSsIpsETlhEuR5lP9Co43uxbeCSvR5p/ta+9yZwDQYn/KY36bFDSLs5GGaAYXSWRi1nbsYniLkPJgKYuqkQ3hIo5ssyrFLp1ggrTw1O2eLmYOEI15BUqXZGEgHU3WQJsGIhxyms0l8vch4DvMoyKCTnPqdZ+nfgToEC7rIihBGRRSPnCAsEKEgDNPFrPhz7nxcOocZPPKcownPSQYvYsA8L2JAAqmBHwKmBFB86Uzx39G4DV0Xzud3J2cHA/9s8PHk0/GFf3h29PugfwH0+izAArSbAEkkcANGIWjAnuNNZEABh74EOOefPlyc+2cnJxf+ye+Ds7Ojw8FnZgXki2CUAGEBoygj+IIRCSIMIC9oNqIkj+6QKwXs85NPjGQT8BhmB/gsmjHpPoeLDH/4uBwBwmfRJbON5FwnmM+j2ZiJJBBR7OnMgAdwaoePbXPJh+k4gk6i+whmB6bQJ7Ju9GaLJLGlAHDlfFGg2BiaImG7A+OfzmGKBZ/G9zhwtbaB3EUapgmThlo6IxjMqx39SZjf6j+vQ/0XMMBNEo/0R3/P05n+O831X/MkKCZpNtWfZYBdajzJb2DYBl750gCDPKj//hrPJ3FiPpJ4kThCwUKMK97K320C9TWdRartPChwULLpKfxU6088nAbFPEkLaObNl/iXA0thnhR6m9liOl/i89ncIAEMFx5i+7ExpmiWp9kkSe/wXTFRCJ0efZDIHKHcafN/Tua5apN/SWCZzzxgjiwOc9nehSUJciBc+iSG2k5wG2XwrT8HJoxz5C1+QZCcVf+MgiQAGTb2bZAk9uJJHBJb+VmEPa8BEKXUglAAambxfduZdCRMhR78AfLOhy5v18HSHpcjvpe/0tAPFqH+kyC3FCW1eRBE/AIky+saePRGNkuCZZTlilWETDk6dPrO1cbTO/TeFX8HLQdnRx8HICzX/vaN+JZ+wzeXO902bFQv2053u/uq7bzc2X7ddl7DP0Nqd/Bh//x8QC2vNoIZbDhBOEvz6GqjDf3dgGJQ3Cz5B/DVxJ/Gsyjj3yA+/WyRF/wLRJ0fghSI7nxseLXB8N+ffDg8+XTh/5B+GOT56YejC/+Po1MiyQH1eQhi0D+fJzFw+puWB2JA0EH2L5ufffD5C1/u5G5Xb8/Az9/vd3df0QedaLQX7rx60+2Mo86bYLT7MprsjXe3o+3d8d7rvd3Jm51xONrZDXZ3O51xOHnZjbrR62i7s9t5Fb6a7Fh4aJB3Xge7r7vdV6+2o264Nw5fh+H2TvhmMnnd3Y46k729N2F3Z68ThW/edDrd0U5nFG6/2n71ZncveD0Zj19rXHJwMTj0LwbnQGfcW5HMDxaZe8AFXYPW8KSzZxMcHr7srl5e+oxA89292unHN68eTQxLZmhGcueVhWN3+8nFbo/g5au2hWR3lzE5+virf370x6DtvN2/OHhPfwMa3e4OLJJX1OT400eNWZNo5opfLBneD/YPQXU5OYVxwPttb2ebnv91cPTre1BqBgf7f4PnnWiTN88anQdevwtA7aD3rMucnp38ZyCRf3h0Vmob0OxYbkertBK93QrNyGi2f7F/PqBFUfteWzS17+lf42iC+hpws49brou7ZatXztU4vkY1uy9VA4/buq2yBanAuKnyp14KKpIL8za62mjhzpcXoPhNe8bsg8rgjNDUQFU1Bp3cTYLpaBz0RGsPdUV3z/nHfwTGabUdgAXQehUOYuw8NjRcgqgQy6Jikc1km5vonv8SqKvBwxbqo5ZD+LcdVKgirS98CuNXIzTeeKxEetMvoAe6/CMndbgN1kacF376hX5an6FqG/kFGCQu9u2NQbnIXeq7DUQZA5g+rHPYmhZZ5Ad5GMd9YrcWGHpAjasZStVoBqolGBT9q41FMdmEHUcb3DwDGwRm4oL2t3ewv+EnxcTzfVAaaE/1cYX9evopF2/Y0PASxHt+s8xBB0j8cXQbh1HuUkvoofXzVNyu53wkiwMVbKGSS/uFlF6YYTQN+aMtBlRq4HHEu3WRLdX00RZ/nabXSQTDA5OqVBnREqJm0X0YzQuhXA+yLM00/sdW+qKJJ+IZW7T0wmrukb0E9NoCeoJ2UWxxX0BjYPwwAh2IWvRrpErLBOVnaVpI7rMBbn1cHjJg9RWgt0IW1eLMahaZaj6QUvbWDKblRfeo9S5yWLiqa7TBTKg8MQBQG8tWudn7H0jcA+MkQBH0CIitsFQooQ/0S4CpC/oGAxviCF0BGGEdssF2tQG/SpWi5cU5i7MWoeVcDg3IAAPpoDqoihaz8zmJrDmKKzUWDyUXLvsWApxjn/xLeiGa9zp3vhbyFaxxI1OYtZz/BHtUFfcsiGHM7wDEcVq8A1YbE1e7tUjBJk9NnAcL+GNpwEvHQu4552BdN3OGp7Nila8U8MttNTQQwgFMvuABJLf+FZJJ+phkE6LX1YZls2vcw/JBLh6X+HnV5mtyNBC6afpW7uDEadZo9O2IHphIrdjp18bJWaUurMYpR1XbR3cSSznGqUatIA6v1TeoA3vCavm67FWo7Ha/deoK9VurxzzZr/ZV2XMYhDfNMpU/9qnRFhCfzDAmVpQAHmnuoYvH6aPaP0ONVO3pVfkJOuxh7+rqvFiMl1dXv92kwYdFMLtIi+PrmziaX12dRcym+pr56Wvh6aWAFH82mzdx+ZNMrjasb+hMzthzqF6yyI9fBTo6P5fvzZ6e5HSbhwxOFwwk+V2wu76V83Zyxq5isZNsSO98vpizT5L81XgA8FdQX9O7/BfyxALYJA7jgrTenHbRALbemyiTvmfPUFoFt/KM6qy75ZT+F2qIhORWBsttOYa7hd3Y19dZdA3bjvhAforDLt+JkZOfPcprWoo3sh0MRHl+QS1wRdu2Y3QHKp+CaNhV8lBjTcsB5nAUWSgJK4JeCbwM3Zde6KbG1Ub6ZZXdMMHjpKQCYTFL4tkXdxrnOUoeAy+hSql5K9Ugm4Gq+shEstGemETdmQ5AJ9i05zwo4I8lpqiOoWWGpDcWcNteZxoiAlmywEoly9SfmnBVpp+0qwTqPSSowrCuzRtqI2bObMDOLli5DEYfiNnwvfBzyab2IOuMsnWOGv/k/B5l8WRpnGmQaY72Fj4s7lKYnlsgy9gRvWle0f3Z0jl/v7+J7jCYE3X65QC7TIMivMEzmjmfmWinFaMIphBP8yYRcH74Mw9OXnpyjOhMntMZUQJshmjgSNv1B3dSTMdojMeFtC7ZRvaFW0CTV0KKQk9T3ImFLU3nJ0wq7Kvhu9vyBNYvcfOxvf49Q4SPyWOQpME4dzVsyGnC67xmfbdKFEwAJVLrfA7rRwzrOkKm1IYbj9HfA/aIcpMDO9AQ1mhry4rf0REit5pzopWjAMgpWZY8JsWCjWCpDrWdh8eWeCgm1mdXFnunAQAQ5I5UbURM8x+vgxszlgO6OvAOSLEJcXUhN2pctRqKGlXkVvwjcFHH+LyAn4+N4KYVyOiuQo33WusQSfBveBPMrlGWTMB8diyknzWRdFLlo06eC4aSLl+T9RrbrcT6AL8S5/EC5ybs8igaS9B8ZGMiUHm/suN3wJyb+El5pFvTfw0PTYNZPIlyxUzUHyN6qZFwCD8qbYfrzKD8CgQmoJPR3lS3BnUuMfc0m6GRUHpre1vD9uahy2o8aTt9g1DlhtktBbncpmyEleD3WUvtg8oB5O+ZUnrC8+jTu0ehNo/Qt3wXxdc3Re7d7MJT1Exo8kAzobl+tDtBVzq5lR4eS10Gv2iXGo2NEXqaprn7ozUZbBWExSJInH71EEDvi1vpggD57xIpgrsVUEVDuWSpy7zIXBxZayg/rUNRm0Gg8YHSIATX95ieku4KsQpNtQ5hRIy0cPTPk3RJmw/B6mMYkSvGYL0ExEtGtr9Dmst5XcmIRzPa0jUAJVMwUxnSTtv+w/y2qh5oVoqtLkjrZ5dkjIfhHRt1y9DsQp9CXfdYSxhoIlv6KUyZ3jC0DKxCtEbHrGoAGjZWEnHro0teeYCXV6R4OOGuKUmF2KpBOCDdM0yn8yQqShdMqZWrKXJOYaf4xZ5HUsEt5viJ5yI7nnMeTCh+LQtYcUUNNbqHH0oo36KSK8I2yuMt8Yk/ydKv0cz/Gs9dqf/weVcRZLCN+GyAYghFWtBW2XY0PUlbt9we5tH4ENiy/LJsivZHlNU0BdPVVsJ0aWM098gYzm3RIQQgd6G5+0GWimcr9VkPJAWQgvhIQ6XJl244P0BIfZoxsTECS80J+VPAXjXwNyRW9YwkL4JrDJ+0iOThsSpR0zWfkwcSjwB98aXtcsd1z28aKNc8qtMgK2LYDLQhZdE/L8Amz8kCjO7QGGfgj3a/stOqP8NsR+fFIsDL+yOe4wZlsCSdGosHVcxxs5xG0xHwFchS0cwDM1IIhl6tm5z9tuJDD7umYL0MVnAQRi4eqMI/pPBu2QMrzzKCjDyy5LTDz1sePaptDCRLk1vaaVw5xVsOfyXeufXdlDs79kZeDvjjcnuInFouMVKbBMFLcHJ3kn2Ls+m813hyULf/fprlKGqgg8UMTS7c3dFiEMQDDrCo+FhHMDkxgpWCJJFkMNtqxKkRCi3deaW9bPZg2dyYRdU11LK1KH5b47hapUbxR5VwA36sOTRJ/RWO2BphbPmuNL8tUoTebr9hzrRjpPCprkszLlKDbu7S0rErnYr3213uthJrhY9Nnbyl+25L06Le1WFbK5rKYgGwdIU68LAJ40bbfwBSCZ6hGBKOHEqCQgoV8fyRMZ3Bdo06a5osprOcI5is5hRRFYyihP/UTeJcEl9Yllcbj7or1AQOXJUvRjBgG39PNKAtC49erfe0Nb3e3tldqeFIj11uGWhx7gh3LYKvajolFbMoBBOXDrkf6mnmggrmGS9aGnF6pEljE3rQWhHmZc0RfcRPWo+NX6G8zzB2UyrOioB4/F4sYFi5ixEz9yJAZmgOsCmCSQu70RQWkyro2M0K/0u0lJ55kZ2QZnkfqN9mLuihg8wjkQRiuFWJNQLGsNFR6nej9T78pokXp5m2xyCLhJ9VWZ/Ik8qZ/jwaa6gJxigP4CrMoktb+QZlbTCCbWpRCKUNpL8H1g1tX6IN7XxPG4xiw7IJgX3DZlXBx9iuKO8DaaGk9FaJgL1NUGNT1dQtLHrd4m1aMvbT2EuPhZDTaCISTqXt+8QI7In1gvFYX1l8KoR8pYlVa5Yv7d+ljBvyYTd+j9ZXmM6XLikwwLDED+44S+fmcZDR2WUpI4ceRbexAQRKKZhysBhCYcvVhb+u5emhLCDqwmHINufb6BB5hXxDL4H93hKBQ28azEV4onPPgsvglvtWaeh+D4nhc+zsBgMfuy1B6QrgGvyt19+KvlSFVlHLgkXr/V4LV2p5qOetcfCHFjYosTR3MvUKDHKxYxkHwkqdIeeBCORCiUV6MMgLKzZdC/2jFB6U/LpKxFq4agTIkTQEQR+NXS3QSyQdVYO8xGB18YD7N0EymdmMlL7EjodPiwTpudS4WhMHFZUbEWaZgUgjEr2qwUUZJhybK+QUGFn4E30Z9t7Lzz0ycZeWmVJup1XRZ5uhoimJdFNINRmk9XTgwaWAThLMc0esfCAFvaiYHzrHeJyx5dapNizOy8dF6up8Aryc+/M0j+/d1hp5KuZK4XXGVNHUxB6x3lrQSu1KcLXH0pYMyLahSTGVH03l31Sh0WR4l6EdpNNGHSUAU9gfcmhh9zVucva7S9X9EDQoDPAIcO0AyWZLd/VRkPLKo6ISxDNDTy2BCalQK8Nx40B7QI+EoAb1lgWHU0vFpZZKdRBlm7WBqjhyHYz8PFiM40Ic5rbB3JAkEtsA2mW+mlPdqqPZtkwf1dLSK9saYKIJ74nYsm6HbWsDNTEw7TsdrGxfA9kUdzpw8lT6YgnjN9s6yCSexgW7LeHV1cZxFGi8wKfxIqcUU0S2iiyKtvIoxxh25+gwJ5NLJQ6zq8pD9Rz7eDR9uyIAzKEZIe9ujzYinBdcCQav0Vp7r8ze2qXyHbEXZ9FoESdjI/iCIqKd6yyY34iY92DssPnuiJMmtT1iejB/MI4m8Swmp12Mu8YcnZNG7rbzWXzexwDwz5QM7gS3aTyWCbqIMHaHy5F2jGMglPiIU2Ep/NBIEMfe8BvSXKfROIYpSJY/K5yDEP0TqEzf/o8AgU71s8Hp2cnhp4Ojt0ew4P72o4Cr3BJY+ddJOgIVDU8IXD5VBKJpQjLNwXK8jbN0BnL19G8X70+O3++fv8fDDVbwyjMtBZuFK6bZeiVctQnO5l7zy2KiXha+1aBs9Sfn//73f/vX/wFsMPq3f/1fzumyuAG+2nKOF9PTJfxX5ZQgV6TOb5g46alO9Cg0/IcSKz3MAs6pX0bBr8FPJGMM6D/AI1aeAu6jBqa8rEFb+JXozKdsuL6LtGcdGap/GfipvBcVYRIkMCmYeu6nc38cgQo4hcWVTzVlqJQn5dsiDlF14jT9HOTi8f7bDziLLT3NpBwZamBR7xnwFrPgNogTREvXeeSHZ1GQI70wvk8bMHD6nqdW8ygIv4xg+belSHl73GYBO02/RKT/m1nMU5WKXZu9LNOWORrs068Yebl/cXRyDNvA8bujX9m7dpNm8Vfc8RN/ksS0BbBDBRSzlGW/Pwkw9I92B297b6WadLXxNQX+0b/obJOmBX0AqxX2q9XARhmKuFnEShpt6o/mSh4Dw4NSXGCi9TxLw4h0FjeeoZDSVvM9ZhtS4rB3FuVhkGCrjrcNa6a7u+sBjqjC9fHQWQDMqBnOqIRmu5QFvOM0mwK8r0QsdxoFs/7ltrezt9vG/EVUFDCN8dVwDf0S/rkNshjzvxFGt/vmxYsuft/t7pR/7cJfawIL7uO8v9mpDk5RC8d3b6Xe0d7nw3zNcrAPOBXI5UgIuVMJvXpGEduKzHWCVTPIiI4wEyx2jvCn66qsUfXXy1aJcxbc+dn1SF9ZweIahQGd4jCo8+ifFxh6FiTupUEZOeck194Bh7s6z1NsJ6DYJzzbzV+eibXg0gpY75s/YCW4zP9rtT8QS2TFN0NBlYnIN/BRVfAFOWSYohYKU2VdXAcNa6akqmothVJJZzDbEnGSnXuHCAdkV6fbgX7CZDFGa23OyrdiFfHf9TjWPnRD9FH9nUf9RkaZp2nCR07BrXHsKtH3iFhUsURPDiYjPQoKDAOHF7K1e98uq+fI3E7YeH+LojkIZdDqblENy5HeuBXkouIHmNMjrEWDJ1yRtudqkucwS+egP7l6lrM+0a7ERg1BqFsaDCS6q6VRt7meDM1JH02SSTEN7ikOgo5grjYmoAYWL7tS715BbikmwDwEXdznzjUJoQk/5oePJBqYydoSWcWkGof4F1KcmBzaLuleVdHkGzpSj1y1RdJvM+qKHrHLjv7s6GlAazIDZRPZgLprAcJNs5JKTVNGjCE/4UmsulnouQHQ5Ud0HA54ZEWO/iKOBb3d9SmJ+8l0Rp5UdJHDCgcosK8w3LZkp7cYcGfsYK1WXTwHehBsJMkIomdiia8c9HrRDmBkpkJyoDVDY30hYN6BYraYsXpUE11Rh+IqhOpoB2CeRao1x7SvJGkJReKVOyW6qyNSKl4bXgX2wkSlM19MXbA2QLiP3VuPxGeLaHFr0KHs2BcT2LIUATydBCVTrD/SA4Acer4MPfNEO7esMdZn8VD+zr39cTD9q0vFcyhLDqzRfpJ906aAvlPCFyyAMFj29eIQjT7BJM1zgRX+CRL6fB5keXQAiFyDQgC63kGGL2ATJh9/EyBR+aePhVZEhR6wB1EmRfPcB7PAL+3oPmhef4/RJUPk0b1Szypz8B1VCZ6oRYAYTGiG+fD1P1CVic5PrzJRUgDdJD5X2NOUXa1H5htcN9Fs7IXIxb7wc2m0uA4xdgDjbrWHYtFomtQa2jX6gbQ8yDIrxsgQcZ+OgqZ1qzJXhJ+ztU5KiQzWoVNYCUAPLKaXsmiQsQmLcKcHtRO3tc30cZ2T4zLASXYnQJdB3DxMEfZmnjUgHuWIRaJBNQi6JodiVdjyGgfGhGDZ85bmiauLym8Mnl5D8bGkL7GvYB3XDlG/xA6GlXVBX5qrYBp8iXzMBJO+dXeCZxOWLEJVlJ5XDiHBdAF9E3aljM6QqSybthDo5ET/WhylDFVjVlhh4wLsX3aN7Gv4H0VxsJDkkBZ7Tu7w1HjixSnzd03EvB5awC0ZrE8PXQDRphO9GWDaf4n5npjp7wczeE82jy7CDYCeSBYCAlwe01Txv18Om3sX51w5bJB8JtV2Lqs2DmwtsOXcpGNYoqMYDB0QPrb2I2PcJl6IBqQABj+F6t8SJKsExU08nGtP+N49dCT5XHfNzxOqLkMUzMX3oCDQWbN58kgkbGMdPgzkC0ACJj4Wg8v7Ev7+p4uTi0/HgxbIUOB9V5WHaoGSEk0ifFhprPvKtGRIJRJwAbGTPbjGIyuuTyMCTkTROC1AfA7LCEluJDSr15WIOC6BIr4SElWLLefFx0XrNupybsumdZkgTUfwlY/FKZYUuzzefs2eVRP2rscXyQ95L6NfOYsOsQPZHbedGr0lDzBeDcDMcPegzbsqN/TQh1arJYuKyDi8OYYjhKCFgbAwhuUy9Ir57QlZ0qgtWqN8Eo4TFGnS70Sbu60n0un5IKjQz1XAOg6usyjKWfkp640qTmLhKtYoUt0cZtvGl9qt1DF+qSoX0r2s8nK2eIphDWIKJp6gMQ4/Kfnhlee8AzJ8NdKHhYtPZBGDScbBJOq0WmUWs/O9/NQXObD6GayVm9oziwhoJ7CVhmUGq36eKRpb2ZS9xuTJmm+1DBjK0K0HUibvPgFCJgnVQ7GScGpAPe/geZ0T7eeeOq/KM+tVNSrzU0q+7HG+UNupSfjq2bk8+vecPUp5oFpARrtEiryhopJLz1GRmVrWEbpPdbeYOJyXQQ49LGGtU0W96HRfi1PsOjaWIdZ6xIHVRE8gB8FYB6AmTwRaahZA7UerdXvUc63P6lTbhqIi2kKn1Sqin35xxiln/cT3Mpk5N6sO5Kpij+7XUNZv3VjaNqpWEQcLmxUSb71Tf61qO6dxmVXJzdP9am2CLGJXsFfmuE6DMEulxwDzvvlsPyZX2SLjHZULtYAqYxQ3dwa3UbbEOrsYjMkpPxyXg2XJuXOKQ8NSIAEqhlgJ4SdWa3iNKBHBUeAjbuKgUtsasfi6IB2Xky6wCHM2xv12iqemDJ3GfXp28tY/OPnw6eMx1a0FEwqH4stAOkePJRSLe2haLOwW4arGrixuzH2AOio8dYYhny8mMEVUH1a9x/Dmidaea/nQyZOW/2bEa8nOONqoFD0idUA+aMkAJmOFieqIZLLbtZnBfCcMHzmcSQsmjK9Ba0HM50nh5YsRFsPOXXiM5kPf3Ws7r0GvmQQhaONJmsGav7uJC8OlGNyDzZHfpHcl+iEo89DyLdixVGrydhrP+tv43+C+36mjCrxwO22HAXj4q9UyekA76L6Iwy85mFMgHPSjCkCwJJU8Xe7vQH83AR70oQFr4YvQlk9CM2MuyddbbV31QYXrNBSIkEANAe02TtsDj/8ya4fDntedPNazEAUbgqjSWrdWBC0SFUI8gqPSzLfmz0kKQgbn+o1NIfeejDNofcrSAlm67SzlUzqbwEKicZHQscwDs6CnpS4+Aj/Kx9VI1vjaK8jhmgRLPMACfRSfofSG/7rrcfQcI4jbznge9zt72wACOZnNAADSqlvaeE4/4jgLV0nC8qRJmypS28aoVlJdrbLppSieQa9qnA1kIGuZ62kmSmPaMOh5DYBRmiYtoy5Tbn2tC7qn+octgBQHs3fcGHxWd+LxfQ0SsZ48Z0Zk01nC+JpOOMHySmAHmWNqJaw6WMedjhWvrJZEZ9taCdMgR/3c1Uj9T32GfRkPW84/GK/+Ub75S4dqg8bw5A0vC71V3+mYroyZqAGAvXl4jmGdR1nRw6OYYhNjKpJNkbyUA4VOPfwlscOHsM8aDwGxlUlRyrvfc2iaXMEFl4jb0MNID8COFj4PDOg7C2YrgeJHvs6RCrZ89gT4R+18O57ldiSxiiCWLbQoWl6lwM5qXfnUpCl4lriH1hRjiROSXRKphwAJwy41Vm05L5xglFML7WzE2XTwSWXow9bq7DLiRWRmTFASnWrcMMpiyiJgzIA4RDH4LyLp8krcRKJFS1PAX9IqA4598cLptjlIpqNvY0bIcB3NhEanxQ1vAJX8zjZRk3gsrI0nByqAwRfzWqYBYGP6o+Rgm6j1cPDuE7ZrDuPgepZyVBpol7+YN9vgPqcwB7FdeFcbjxXFguSDoVqAdOhSlTxWMDrwe8fb1ai0yCkAB0d8Sf8SbPFPjlZEFiFfbg89hOleotCBtSj/e7WxuYnkkxrLdRYsTQVA+xi7q+WitiNeGcdxVxvpZh0s3i/jaZ8kYAu3Se1HuZMeaD1oW+l+2UX9Ki+32LMoiVkrhwkZHAz6D8ARPe/l5LGCVGfo3WCKvCYBhKCBYSBh+ySwYJaSOagI3mvhjCUsxbb0NMh/WRfm0SxshmpoGw00OhJGslQ49IboPIOlM1pwUHm1gyS6RskuFAxT6TD0kYruQZkkkuYah6yld2g+cVy5vCLrNBK8yWSePaGM/Ihd/Hs1iaoWsPZKhx/6Oie9ADeGdmmORdAxxr9Gbr3SDBwWgIEIyoIgRZ8htOQhkTFU4djgj3jPp/KR2xzXz89rYy1MXcDM5kE/EJvWMhla76DFEa50Sw198tTWLXQC+1of7Vs+4ywWOYvlT8eHg3dHxxhn/Fjd6DAeNp7ZMUNz2AUK/JeP9VPlpTkC8TbzxGVP7NNDu6S3QEnexcNQ6m/4WQ9keYNPucsa9/o0ArGgzMvPG25FWhPQd033KqVM4wPt0qKG2Q7m9kyf/FaZYmPnUtPKIhItMLa5HsruKluDLhQJipxUbZ6r4Ob1W8z3bsCG2Cf10MEUPMovz6j8lWVuVl/LveDs5MA5+f3syc3ljIZrAD5V06DgnZ7Z4KiW8T1Vobm3UxrBYBY7TGlO78GOAI+vs3jsit2w+w2mL3PRPPPT22ztLaequdco6wpuqXc2qesqqgh5yieHv0gr5kVCZ+9tUdVOHEZVti9ZjrLfGBJQNkUJw+egYp9yRUOvgE9dVKlgDY1g4P1tI64k4+htKjQNZtmEfZLGvRWGY5Ot1yAPMmBRt7ZLvSs9ji6Y5019gX0gcCnbL31ywz4dphBbBWtMPyyFAqCz3aUU5IzzQ3UrBDVzcRwa55QiFrkGjErWdqN7/lQ5idnpQjdG3mUpxikTItTXbJO74Riz3IqZsY9mrfGAFBWGEjkO5AHqzjPxMydVHBsAcLz7EWOANZyWKHfHXCZdwyTIrtHtKJDRPLvSjQ5Kj5mce8+xjyQSpMJCUULlwtCrKbCtbSWvazd+8L17eGtG3X18LvNPW2Av4yYwqCWo8WCuG/4o6iRRqVTlbuaVS9UaRBmUr1GG1RtuSUzqS46x84kRKy4DetfyLowQKos4eImtKjRkTboG3EvS8LLmwrI23kw31yU4wpQy/mpj0tkkoQUrbciLHXSqWvTr/Bl1U2G70MOp069chvi8+dKB+ejt/W6I2jlFn00EHWXjgCOcqsMNy0VjtSPUVGOz1ptWGEaKuEptiidCuTSbxOalB4MxfvAJvtDBsTxWAQbqIpMeEJVmwMxUXsaBGwW3KRPtZB5CGwuob4ooRrsPmR4uLuRCEG55PxK+/H37DQDP8M4oTCN6Qpw8ucxrqnn5WOer1YRY/Qn+ulLNPsfX4gMoCEHTGGSJz55QGmTUH0+S5Jzaul7lS6sfMoTLQgpSWF/GQ+WC5jU0tL+cy/ONJz/HlsPajjULvOfItaqDthrQIrYAlR6SXqkv9Mv1bp449KztS9u7FNTH77K2dReBfYo6tPdPZV3VLeca2aq/XeUm1tvhHvAvTScow5pOItQUcMOInvRFt0vH73pnRJaQXOHA0cLeKf+JtyHMByll+irNoubi1ea7VluaqT3BMfriUk5x/CHc2BKTf3BcqQ71rdIj2nWeLd2JLaMO9IiqnyaT1xQdpX9bKMQWFAqXEOuPdHTZvFTuWpqdH419taA5KqdWGJTNK/KgesJj3rlsbeWtBoQN9wDDsR0cDTpBidq35sE48qLpvkAG+cxWAhvQltqXxFm/wfmnIPztqE46Ck15efUPRfFbUKtc1K1wbLjD+0l+wswkBWUTz2vTa5QFIWYum3Jc6ZO4QhgyujXl1omX5L7GU42WPEq0ezPkjlpwxmMKyxPpoCr6DSWUeFrrX7NhsxOonEFD3m05qyHVHKytPEarVxk4NRo25jFs7ZSrjOohpxfP43mEcfUEnt0FNpQyaNWfco4XK0EUzd4hNayzvb3tvCjdDVsrxNx6oZJa8ommKKw+o7TPJcXvavYHP1c+I1z42oah3X9gO49oq3OlWtl27LI8rm0qtp1qYZ7eqmMK+56GXl30lB5bWZP6pNC2QhhE4XNd9VAhhrIY61gPztROKOyi5tUTCZyVsZmmtU7V8+r5QMyZgWN5SY8VHWncJKLHDNeFUTbU3q6vNX4gaeA8KCI+6slK5FzSr6/RrxWoyZS0stYa2bX1DTRC79FKZbW1ara+iwpaJV+t1zo6qEIwC2pespm6WmxWumXNz6tZwGLgIlRLRgjzxVizpaCFKo74rFGXdeZVYG1ZZx7jhjiouicDxWoLjAsEnnHvdskujZktVbqY7PUN3u5qMin0rZIx9DfPTcyw5QFLaRYDHCrSsJh7T6/ktXSa79phVsJdR81fE0O5Bvyg8BcFnb1hGi856WfpnYt/fMUkdXiJizidoJ8MaPzYfChY2e5goVxCA3VPiV62Gr8c1txX0ujGVvc/GQkvdglv0ifVfnqpaDVcHazV8JE0l83rMmnb04VT7Y0zNWa0lEZWsHslPccxx/WTosr3PGdfjgvPAWCZxKKyIGjeeAPSPMBszBqFRBjdUhGx9Q6jFra8tbeBxJbRzH52VEt0G7FdZ/C1K+ZU27BampdDrQHRLo2AdoMSvQqgoSrXa8ntNZTZoXaoghOytNkWK9IzpfTjd7QydGuSaM61destkCYhQ8l7fj5uApYXY3c8TicYtLcWRK6KlEtvAq33VunKZBpIGUZV87k7L0Y/1vZQhh4+PtmXnrKgs9KwZZN03UVN4W4w4NWuN8Cfk0Geo0mXB1n6ZQ5ratXO1jp0X3EsI0aCtzHox1Q2ESu1o0udSg5ZCwhZ0wn13KK/1cgSMRDkjkrsybC1HkjDaVQDUHs/XA+i8ufUgJMv14TFDpcaONox3WM9AxqSonxR03bFEijb+MZiGC1pma63FrzrLF3MR0u3nPHW5VNHkOSdBnzcSynOONaH7p6rTeZZH3lzJa+f2fPGjruFDVwIZSFLLuV/lVQestu6Yf9q2Krqt6RhS4/mBT0Yneex3V/b0R8GxLXOMsqyvnquCfYhXm4zpwHuVtJZVESut73b0gKAzpnz9EwWjcI9Bzt2/s//dkSS+vmhg4PLc06Xk5d9rh3bo985XieZSY0Q84qSMh+vHfgDezpJ3pJZ5/FtWoirPEqGVQllUoqJ+kD4AKcMTWY+BahmQj2XrWRlCpUlhkg+lSO2OvdL7bYwjfzH92R32elO54ImDewA1N1kqfSuYweIfUuSmDaa6oZE84KVkoBorDiAsRcO18gloy/1LDItIIirQcmExOOt/VUK4Kr8MZPrKd4PV7KoFAIzGdyrEHCLXN+1TJQ0nHRQlS2AndZeJqQKqKQ6VPhFcUlyIVg39n17budT4dG61liuoO7rtrNrLCEVfdguFS+62PEeQ+0ZQJVzRMjWT9PDCKsnU01Nhawu6kPj4J+cRqr19Ox00t3GdFIL6nMEj/UpyRtXTExNCadqFtMaSabPzkr9d85OfWaW6l7LXl1N2Rx2RsdqeaLYWPKvdHhYQsUMkWhINb2m4HQrgqm0GLgwxQqDmw4/dlm86SeVupuhvKXpUle2tHDS1qoe5NnJzlOdmM6P5s6E1g4Db9K/rzbG6RSLC0MbfxzlYRbP6d6vJrWbtAwEqG6fkjRsUdw5KY225mFlXsEC7mvlwGV499FM1GAgR9VtXinK4LjqIII3e2GKteSKlOuddjfA7CMVZaA9jiK/md/quFO9rOfNRkqt2N9KmKY3TjnBCizIyR4w0du/jyfujeeckb3XVne10akC30aZ/yILa+k1L0TZZ9rp3n368AEvJNXL1I+CPHq1oz/5msQjtf5wv6OA1AlW47sCPB6MYBSt2ruq9rFJRTREBo0o5DJxHsrglEfPOU7L2k90HVxZQRH+HzlpS3AP1wKrXkdNdVGczw/1RX0eP3vOoXWB+INVqeZRgBbXock77j8/6PF6j59/Kfm4q7Ux4/SgO646AohTSRpxC0+b7RqzbIgoPUIValQ7GKDZrixjUtylNWVKGtaU53y2Ans/G8VR3nXwWGwxoztPgCgjMDQRHJAADw0xpILRkPDU9J6Csk6EFBwK1Ce158UWis//qou3rc/aJUWMdLssNSJWlQpQpSJdRCgjfqLtUMxZ3hZejrytInmdg/Pft06Pf2VxpKfPDg4GW28p/AyXxtnJwdbpGcatb97mmxkWGm30ZmtubL7j3PlcCl4YEDb5LNb7FrAX2K5AyRWGbFuUZRZrC+Z5k1rzAWOZbxmj/ISV4zlaED4W5YVm0AH5fJ0HVY9P3JgqIyQoF8SRV8/wLfI4XP7uLsimm4u5uKfiJl0kY8JqxIcHeDEQD5iRV0w1jm6xuKDDB8lqJSVAjFm4BACz8AZPzDQG+YzyxQN1Gog1HudU6o/IYd24aZTkAbzsumy/MLT94/0Pfzs/OmeIJT+VzID51aM0/WIIYuQlqutDR5WUYBbri0mgIDFiPzKMGyMf02wJD6bUy2IGEOiacjFA/j/807il6mywf/hx4E3H1pXLLDaba/vW33mlNgjOXqu99uoHl1yTUpiDsM8Gn84Hhz6ABPIIpxseQ/msImGTg5OPpx8GF4O6Ix3r/Y+tzvbdp7KPsuIinZiP+fhIm8H2qiloqg/WbrqrrKIzPnHZ2pM3p9UAVAKq4Mt36w4YK82sI4uVcPX5XQO+0dzsR90/2XDk8a2Hh3xUIub0L1yoSjPIt3TrZKuueFWzReVWI7Zq461Xn6SssNhqrP4suNPA1r1l5fV5UJVtuQK41midPlbU21iRFLkS5opYL3k9tQjpgXmm6dMPo0ou0AKPhHCl+x+NK4htPCqNUQC4+B8fzSKM6xZlM+qFNy1+X6LgY6QOXqX0rUK8BGTEWcunhmAsb3/tyYAnOwGaC+9JskgSCk/bu/2jD2V+NAbR8euVdVepgIYgthZrUQ66BOI8iL/My8SOTy4Gb09OfvPFnvTHh6O3/ttXO+QcFBut3OCJwmapRvmqpqUIbKoJZSpb4RXBmDqgw+fSwXztDl6GUlZ/J1nnb3feePF8ORuJHsV2RvqHET2H1hPVrJ6CmMhzl+0rb/RqR9THbhq5dK+RhwlsR+1OcewFEzTgaW7cbg6ThQ/FO7I2y2uRQ06sMrhcNJY1NzHa5RLjaj0QVwnWnPrzE/Py52dFlU9Wg3toevuIHdllp2quHpo4hLx2+cqT+AtPPQUTr4ZOF5NAM5wHm27DKj813AChzWDd7Q/l3RCdlvMXxPCKJMXKGyEapA8oacs8zpXSUCd2hKtAz5q72rjtSNVnpUyqKnMrFMz/ALV8f0It3VUFbe1Suc8ptytkr3b3sHZbN1WqJWFA92yLSfcyvMUN9JMXGHlriAQqMqt2OrZt4RHtkurqh3dHHwb+x/3jo3eD8wulFBy8Hxz8dv7p47kn0X60q8wJBlNxHTKfUJS3VfdCC2SfcSU0kBI2W3+0LDiRgUCaO7F5i7POCez7EX47sxqoCGyvu7C4lhAVH6rxSZVIhvWnsQytaO/vMOW4Zz5grOOf+Zs/Dx8dhx8g3vBTaKUi+lFdKL2uZAgSLKuHtyGwLOALJNXUatpZae9ITyj8VdwLTdvTr+CQ257KOOBU97Z0LXIIcM5pB9JCEBfG45BNd6Gvm/XYg7gFZ519ZQ3o0gWBkGnn14s+Rejg5MAMHoOHm4bPpxSwlPBDr5jOjQRvdE7Bc+Qv7494jvcBuCUkJNAdhwHwfo8HXGXro1P/cPDuw/7F4JCu2hHSt3qE/bwlru1/NUvdtaZFgPd4lOVtKlh8xWKXhhMsgTXzt7jeA57RDXIuXSxkzEFr3RNP859G0aHLjkpFKnOsvdrzvRUBdvVDrL+g5ZsHrXZCvdZ204VAI5Ss4qoY72YXBUvT+KuY19wQ8Q1INyLcfLnGc5DU9vwfjZ6mkIiLiWXiyvroabXNfyxymjKjyt9rjqR6JFfLn2axAovDlQNDHwpGObTQoYwKAN58U2MBy0AES6huSUV368FUeaUrRegUsjt8nGBxwzVv34Me9/lTrA9U+lCcSQBjHYMFWQ7XOMMvn5bWCwv16v3OdDUNXupeUgpDksQuhuMVu4Fl8G853Rcvuts9rzN5dD7Gb7+rcv2hcX+mLNePbuK89EbTdY0RqPc5FaDXWGhLBsjDDzCvPeeo4Gx5Pq+QrpJqsoAyyOfJIucTQHwvd2E8Dsw9h64oQYc9GzZRkG2OF3yRbOSQY4O+Mo6a5BUidIABm1+BLgX0+fORK503E72GBmHwGhU8aJlHIf4ax6CoB0vyzgExxX3pLwGLJJhdL4JrfDqnp4gZt+LfLzceVSs/nk1SBGg02XiEJrMRZ6Zs9HbUD0z6T7ON3u7j/wdUeh1i"
notebook_dir = out_dir / "notebook"
notebook_dir.mkdir(exist_ok=True)
notebook_copy = notebook_dir / "densenet121_transfer_5seed_019.ipynb"
source_book = json.loads(zlib.decompress(base64.b64decode(NOTEBOOK_SOURCE_ZLIB_B64)))
for cell in source_book["cells"]:
    if cell["cell_type"] == "code":
        cell["source"] = [line.replace('NOTEBOOK_SOURCE_ZLIB_B64 = ""',
                                       f'NOTEBOOK_SOURCE_ZLIB_B64 = "{NOTEBOOK_SOURCE_ZLIB_B64}"', 1)
                          if line.startswith('NOTEBOOK_SOURCE_ZLIB_B64 = ""') else line
                          for line in cell["source"]]
notebook_copy.write_text(json.dumps(source_book, ensure_ascii=False, indent=1) + "\n", encoding="utf-8")

save_json(out_dir / "analysis_manifest.json", {
    "archive_version": "v1", "experiment_id": EXPERIMENT_ID,
    "source_experiment_id": SOURCE_ID, "source_config_sha256": sha256_file(config_path),
    "source_validation_lock_sha256": sha256_file(lock_path),
    "split_zip_sha256": SPLIT_SHA256, "holdout_zip_sha256": HOLDOUT_SHA256,
    "class_order": CLASSES, "seeds": SEEDS, "checkpoint_sha256": checkpoint_hashes,
})
manifest = []
for path in sorted(out_dir.rglob("*")):
    if path.is_file() and path.name not in {"FILE_MANIFEST.csv", "CHECKSUMS.sha256"}:
        manifest.append({"path": path.relative_to(out_dir).as_posix(),
                         "size_bytes": path.stat().st_size, "sha256": sha256_file(path)})
pd.DataFrame(manifest).to_csv(out_dir / "FILE_MANIFEST.csv", index=False)
(out_dir / "CHECKSUMS.sha256").write_text(
    "\n".join(f"{row['sha256']}  {row['path']}" for row in manifest) + "\n", encoding="utf-8")

allowed_analysis = {".csv", ".json", ".md", ".png", ".txt", ".sha256", ".ipynb"}
for target, include_models in ((results_root / f"{EXPERIMENT_ID}_ANALYSIS.zip", False),
                               (results_root / f"{EXPERIMENT_ID}_FULL.zip", True)):
    temporary = target.with_suffix(".zip.tmp")
    with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(out_dir.rglob("*")):
            if path.is_file() and (include_models or path.suffix.lower() in allowed_analysis):
                archive.write(path, arcname=(Path(EXPERIMENT_ID) /
                                               path.relative_to(out_dir)).as_posix())
        if include_models:
            for seed in SEEDS:
                archive.write(checkpoint_paths[seed], arcname=(Path(EXPERIMENT_ID) /
                    "source_checkpoints" / f"seed_{seed}" / "best.weights.h5").as_posix())
            archive.write(source_deployment, arcname=(Path(EXPERIMENT_ID) /
                "source_checkpoints" / "deployment_model.keras").as_posix())
            archive.write(config_path, arcname=(Path(EXPERIMENT_ID) /
                "source_checkpoints" / "experiment_config.json").as_posix())
            archive.write(lock_path, arcname=(Path(EXPERIMENT_ID) /
                "source_checkpoints" / "validation_selection_lock.json").as_posix())
    with zipfile.ZipFile(temporary) as archive:
        if (archive.testzip() is not None or
                f"{EXPERIMENT_ID}/notebook/{notebook_copy.name}" not in archive.namelist()):
            raise RuntimeError(f"Archive verification failed: {temporary}")
    temporary.replace(target)
    print("Verified archive:", target, f"{target.stat().st_size / 2**20:.1f} MiB")


Verified archive: /content/drive/MyDrive/Cashew_Leaf_model_result/training_results/current_dataset/EXP-DENSENET121-TRANSFER-5SEEDS-019_ANALYSIS.zip 3.7 MiB
Verified archive: /content/drive/MyDrive/Cashew_Leaf_model_result/training_results/current_dataset/EXP-DENSENET121-TRANSFER-5SEEDS-019_FULL.zip 220.8 MiB


## Done

EXP-019 reads EXP-018 and leaves its checkpoints/results intact. It writes
complete Test and holdout artifacts plus FULL and ANALYSIS ZIPs. Check the
near-duplicate audit and class support before interpreting the domain gap.
